# 03 — Contratos, coleta limitada e manifesto

Execute primeiro. Instalação no kernel atual, se faltar algo: `%pip install requests pandas plotly pypdf`. Para gerar/abrir notebooks fora do editor, instale também `jupyterlab`. Nenhuma dependência é instalada automaticamente.

Limites: 120 chamadas, uma por segundo, três tentativas somente para falhas transitórias, 80 MB por arquivo e 2 MB para JSON. O manifesto é congelado no primeiro ciclo sem erros de rede nas consultas de seleção. Para repetir a mesma amostra, não o remova.

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

import importlib.util
from collections import Counter
print('Python:', sys.version.split()[0], 'kernel:', sys.executable)
print({m: bool(importlib.util.find_spec(m)) for m in ('requests','pandas','plotly','pypdf','jupyterlab')})
probe = Probe('03', max_calls=120)

Python: 3.14.8 kernel: c:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\python.exe
{'requests': True, 'pandas': True, 'plotly': False, 'pypdf': True, 'jupyterlab': False}


## Contratos OpenAPI e divergências com o catálogo local

In [2]:
import csv, json
catalog = list(csv.DictReader((root/'docs/catalogo_recursos.csv').open(encoding='utf-8-sig', newline='')))
specs = {'camara': probe.json(api_camara('api-docs'), limit=8_000_000),
         'senado': probe.json(api_senado('v3/api-docs'), limit=8_000_000)}
summary = []
for house, response in specs.items():
    if not response.ok:
        record('03', f'contrato_{house}', house, response.url, 0, 'OpenAPI acessível e GETs comparáveis ao catálogo', {}, 'INCONCLUSIVO', response.error)
        continue
    ops = [(path, op) for path, methods in response.data.get('paths', {}).items() for method, op in methods.items() if method.lower()=='get']
    previous = [r for r in catalog if r['source']==house and r['resource_type']=='api_endpoint']
    # Compare stable path suffixes because base URLs differ across providers.
    new_paths = {p.lstrip('/') for p,_ in ops}
    old_map={r['route_or_url'].split('/api/v2/',1)[-1] if house=='camara' else r['route_or_url'].split('legis.senado.leg.br/',1)[-1]:r for r in previous}
    old_paths=set(old_map)
    changed = sorted(new_paths ^ old_paths)[:10]
    param_diffs=[]; schema_diffs=[]
    for path,op in ops:
        old=old_map.get(path.lstrip('/'))
        if old:
            new_names={p.get('name') for p in op.get('parameters',[])}
            old_names={p.get('name') for p in json.loads(old['parameters'] or '[]')}
            if new_names != old_names: param_diffs.append(path)
            content=op.get('responses',{}).get('200',{}).get('content',{})
            schema=next((v.get('schema') for v in content.values() if v.get('schema')),{})
            if schema != json.loads(old['response_schema'] or '{}'): schema_diffs.append(path)
    deprecated = sum(bool(op.get('deprecated')) for _,op in ops)
    summary.append({'casa':house,'versao':response.data.get('openapi') or response.data.get('swagger'), 'GET':len(ops), 'GET_catalogo':len(previous), 'depreciados':deprecated,'rotas_divergentes_exemplos':changed,'parametros_divergentes':len(param_diffs),'esquemas_divergentes':len(schema_diffs)})
    record('03', f'contrato_{house}', house, response.url, len(ops), 'GETs, parâmetros e deprecação confrontados com snapshot local',
           {'GET':len(ops),'GET_catalogo':len(previous),'depreciados':deprecated,'rotas_divergentes':len(new_paths ^ old_paths),'parametros_divergentes':len(param_diffs),'esquemas_divergentes':len(schema_diffs)},
           'PASSOU' if not changed and not param_diffs and not schema_diffs and len(ops)==len(previous) else 'INCONCLUSIVO', 'Revisar rotas/parâmetros/esquemas antes da coleta' if changed or param_diffs or schema_diffs else '')
display(pd.DataFrame(summary))

,casa,versao,GET,GET_catalogo,depreciados,rotas_divergentes_exemplos,parametros_divergentes,esquemas_divergentes
0,camara,3.0.1,79,79,0,[],0,0
1,senado,3.1.0,157,157,42,[],0,0


## Arquivos, redirecionamento, falha e resposta vazia

In [3]:
files = {
 'presenca_camara_2026':'http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2026.csv',
 'votos_camara_2025':'http://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-2025.csv',
 'temas_camara_2026':'http://dadosabertos.camara.leg.br/arquivos/proposicoesTemas/csv/proposicoesTemas-2026.csv'}
file_results = []
for label,url in files.items():
    r = probe.fetch(url, method='HEAD')
    file_results.append({'arquivo':label,'HTTP':r.status,'URL_final':r.url,'Content-Type':(r.headers or {}).get('Content-Type'),'Content-Length':(r.headers or {}).get('Content-Length')})
    record('03', label, 'camara', r.url, 1, 'Arquivo anual concreto responde HEAD e redireciona corretamente',
           {'HTTP':r.status,'bytes_declarados':(r.headers or {}).get('Content-Length')}, 'PASSOU' if r.ok else 'INCONCLUSIVO', r.error)
display(pd.DataFrame(file_results))
invalid = probe.json(api_camara('rota-inexistente-validacao'), limit=200_000)
empty = probe.json(api_camara('proposicoes'), {'siglaTipo':'PL','ano':2099,'itens':2}, limit=200_000)
display(pd.DataFrame([{'caso':'rota inválida','HTTP':invalid.status,'erro':invalid.error},
                      {'caso':'intervalo futuro','HTTP':empty.status,'linhas':len(rows_camara(empty.data)) if empty.ok else None,'erro':empty.error}]))
record('03','estados_erro_e_vazio','camara',f'{invalid.url} | {empty.url}',2,
       'HTTP não 200 e resposta 200 vazia devem ter estados diferentes',
       {'HTTP_invalido':invalid.status,'HTTP_futuro':empty.status,'linhas_futuro':len(rows_camara(empty.data)) if empty.ok else None},
       'PASSOU' if invalid.status not in (None,200) and empty.ok else 'INCONCLUSIVO', 'Não converter falha em zero observado')
wide=probe.json(api_camara('proposicoes'),{'siglaTipo':'PL','dataApresentacaoInicio':'2023-01-01','dataApresentacaoFim':'2023-12-31','itens':2})
narrow=probe.json(api_camara('proposicoes'),{'siglaTipo':'PL','dataApresentacaoInicio':'2023-01-01','dataApresentacaoFim':'2023-03-31','itens':2})
display(pd.DataFrame([{'janela':'ano inteiro','HTTP':wide.status,'erro':wide.error},
                      {'janela':'trimestre','HTTP':narrow.status,'linhas':len(rows_camara(narrow.data)) if narrow.ok else None}]))
record('03','limite_janela_camara','camara',f'{wide.url} | {narrow.url}',2,
       'Filtro de data anual é recusado; dividir coleta em janelas de até três meses',
       {'HTTP_ano':wide.status,'HTTP_trimestre':narrow.status},
       'PASSOU' if wide.status==400 and narrow.ok else 'INCONCLUSIVO','Respeitar limite da API, inclusive na paginação')
by_year=probe.json(api_camara('proposicoes'),{'ano':2026,'siglaTipo':'PL','itens':10,'ordem':'ASC','ordenarPor':'id'})
year_rows=rows_camara(by_year.data) if by_year.ok else []
mismatch=sum(str(x.get('dataApresentacao',''))[:4]!='2026' for x in year_rows)
display(pd.DataFrame([{'linhas_ano_2026':len(year_rows),'apresentadas_em_outro_ano':mismatch}]))
record('03','ano_vs_apresentacao','camara',by_year.url,len(year_rows),
       'Campo ano da identificação não substitui dataApresentacao no indicador de iniciativas',
       {'linhas':len(year_rows),'data_de_outro_ano':mismatch},'PASSOU' if by_year.ok else 'INCONCLUSIVO',
       'Filtro de apresentação explícito obrigatório; registros de projeto podem ter ano de identificação diferente')

,arquivo,HTTP,URL_final,Content-Type,Content-Length
0,presenca_camara_2026,200,https://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresen...,None,None
1,votos_camara_2025,200,https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-2025.csv,None,None
2,temas_camara_2026,200,https://dadosabertos.camara.leg.br/arquivos/proposicoesTemas/csv/proposicoesTemas-2026...,None,None


,caso,HTTP,erro,linhas
0,rota inválida,405,HTTP 405,NaN
1,intervalo futuro,200,,0.0


,janela,HTTP,erro,linhas
0,ano inteiro,400,HTTP 400,NaN
1,trimestre,200,NaN,2.0


,linhas_ano_2026,apresentadas_em_outro_ano
0,10,3


{'notebook': '03',
 'check': 'ano_vs_apresentacao',
 'fonte': 'camara',
 'url_parametros': 'https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano=2026&siglaTipo=PL&itens=10&ordem=ASC&ordenarPor=id',
 'coletado_em_utc': '2026-10-09T14:21:50+00:00',
 'amostra_n': 10,
 'regra': 'Campo ano da identificação não substitui dataApresentacao no indicador de iniciativas',
 'contagens': '{"data_de_outro_ano": 3, "linhas": 10}',
 'estado': 'PASSOU',
 'motivo': 'Filtro de apresentação explícito obrigatório; registros de projeto podem ter ano de identificação diferente'}

## Amostra congelada: 2026 dirigido + 2023–2025 estratificado

In [4]:
if MANIFEST.exists():
    manifest = manifest_rows()
    print('Manifesto existente preservado:', MANIFEST)
else:
    existing = list(csv.DictReader((root/'docs/amostra_projetos.csv').open(encoding='utf-8-sig', newline='')))
    manifest = [{'fonte':r['fonte'],'entidade':'projeto','tipo':r['tipo'],'ano':'2026','id':r['identificador'],
                 'url':api_camara('proposicoes/'+r['identificador']) if r['fonte']=='camara' else api_senado('processo/'+r['identificador']+'.json'),
                 'selecao':'caso dirigido 2026'} for r in existing]
    manifest += [
      {'fonte':'camara','entidade':'evento','tipo':'presenca','ano':'2026','id':'82870','url':api_camara('eventos/82870'),'selecao':'caso dirigido'},
      {'fonte':'camara','entidade':'votacao','tipo':'resultado','ano':'2026','id':'2611313-34','url':api_camara('votacoes/2611313-34'),'selecao':'caso dirigido'},
      {'fonte':'senado','entidade':'processo','tipo':'vinculo_votacao','ano':'2023','id':'8361684','url':api_senado('processo/8361684.json'),'selecao':'caso dirigido'},
      {'fonte':'senado','entidade':'senador','tipo':'filiacao','ano':'2023','id':'5672','url':api_senado('senador/5672.json'),'selecao':'caso dirigido'}]
    selection_errors=[]; missing=[]
    for year in (2023,2024,2025):
      for typ in ('PL','PLP','PEC'):
        c=probe.json(api_camara('proposicoes'), {'ano':year,'siglaTipo':typ,
                    'dataApresentacaoInicio':f'{year}-01-01','dataApresentacaoFim':f'{year}-03-31',
                    'itens':10,'ordem':'DESC','ordenarPor':'id'})
        if not c.ok: selection_errors.append(f'camara {typ}/{year}: {c.error}')
        else:
          valid=next((x for x in rows_camara(c.data) if str(x.get('dataApresentacao',''))[:4]==str(year)),None)
          if valid:
            manifest.append({'fonte':'camara','entidade':'projeto','tipo':typ,'ano':str(year),'id':str(valid['id']),
                             'url':api_camara('proposicoes/'+str(valid['id'])),'selecao':'ID recente com dataApresentacao no ano'})
          else: missing.append(f'camara {typ}/{year}: sem dataApresentacao coerente nas 10 linhas')
        s=probe.json(api_senado('processo'), {'sigla':typ,'ano':year,'dataInicioApresentacao':f'{year}-01-01','dataFimApresentacao':f'{year}-03-31'}, limit=2_000_000)
        if not s.ok: selection_errors.append(f'senado {typ}/{year}: {s.error}')
        elif isinstance(s.data,list) and s.data:
          item=sorted(s.data,key=lambda x:str(x.get('id','')))[0]
          manifest.append({'fonte':'senado','entidade':'projeto','tipo':typ,'ano':str(year),'id':str(item['id']),
                           'url':api_senado('processo/'+str(item['id'])+'.json'),'selecao':'menor ID da lista jan–mar'})
        else: missing.append(f'senado {typ}/{year}')
    if not selection_errors:
      save_manifest(manifest)
      print('Manifesto congelado:', MANIFEST)
    else:
      print('Manifesto NÃO gravado devido a falhas de coleta:', selection_errors[:5])
    print('Estratos sem linha:',missing)
project_strata={(x['fonte'],x['ano'],x['tipo']) for x in manifest if x['entidade']=='projeto'}
missing_strata=[f'{house} {typ}/{year}' for house in ('camara','senado') for year in ('2023','2024','2025','2026') for typ in ('PL','PLP','PEC') if (house,year,typ) not in project_strata]
coverage_years=all(any(x['fonte']==house and x['ano']==str(year) and x['entidade']=='projeto' for x in manifest)
                   for house in ('camara','senado') for year in (2023,2024,2025,2026))
record('03','estratos_manifesto','ambas',str(MANIFEST),len(manifest),'Ambas as Casas nos anos 2023–2026; tipos ausentes explicitados',
       {'projetos':sum(x['entidade']=='projeto' for x in manifest),'estratos_ausentes':missing_strata},
       'PASSOU' if MANIFEST.exists() and coverage_years else 'INCONCLUSIVO',
       'Amostra não substitui cobertura histórica; estratos faltantes requerem investigação')
display(pd.DataFrame(manifest).groupby(['fonte','ano','tipo']).size().rename('n').reset_index().head(30))

Manifesto existente preservado: G:\Meu Drive\Projetos pessoais\Dashboard Lesgislativo BR\docs\validacao\manifesto_amostra.csv


,fonte,ano,tipo,n
0,camara,2023,PEC,1
1,camara,2023,PL,1
2,camara,2023,PLP,1
3,camara,2024,PEC,1
4,camara,2024,PL,1
5,camara,2024,PLP,1
6,camara,2025,PEC,1
7,camara,2025,PL,1
8,camara,2025,PLP,1
9,camara,2026,PEC,2
